In [179]:
# criar o navegador
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import os
import pandas as pd

servico = Service(ChromeDriverManager().install())
navegador = webdriver.Chrome(service=servico)
navegador.maximize_window()

In [180]:
# abrir a página index (entrar no site da busca jurídica)
caminho = os.getcwd()
arquivo = caminho + r"\index.html"
navegador.get(arquivo)

In [181]:
processos_df = pd.read_excel("Processos.xlsx")
display(processos_df)

,Nome,Advogado,Processo,Cidade,Status
0,Lira,Alon Lawyer,PC6592,Distrito Federal,NaN
1,João,Lawyer Alon,EB3792,Rio de Janeiro,NaN
2,Amanda,Amanda mesmo,MM1043,Rio de Janeiro,NaN
3,Carol,Amanda,PC5197,São Paulo,NaN


In [182]:
print(processos_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Nome      4 non-null      str    
 1   Advogado  4 non-null      str    
 2   Processo  4 non-null      str    
 3   Cidade    4 non-null      str    
 4   Status    0 non-null      float64
dtypes: float64(1), str(4)
memory usage: 292.0 bytes
None


In [183]:
# Arrumando o tipo da coluna Status
processos_df["Status"] = processos_df["Status"].astype(str)
# Tirando a linha vazia
print(processos_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Nome      4 non-null      str  
 1   Advogado  4 non-null      str  
 2   Processo  4 non-null      str  
 3   Cidade    4 non-null      str  
 4   Status    0 non-null      str  
dtypes: str(5)
memory usage: 292.0 bytes
None


In [184]:
from selenium.webdriver import ActionChains

for i in range(len(processos_df["Nome"])):
    # abrir a lista de cidades
    botao = navegador.find_element(By.CLASS_NAME, 'dropdown-menu')
    ActionChains(navegador).move_to_element(botao).perform()

    # selecionando o cidade
    navegador.find_element(By.PARTIAL_LINK_TEXT, processos_df["Cidade"][i]).click()

    # Mudando de aba
    primeira_aba = navegador.window_handles[0]
    segunda_aba = navegador.window_handles[1]

    navegador.switch_to.window(segunda_aba)

    # Preenchendo campos de pesquisa
    # Nome da parte
    navegador.find_element(By.ID, "nome").send_keys(processos_df["Nome"][i])

    # Nome do advogado
    navegador.find_element(By.ID, "advogado").send_keys(processos_df["Advogado"][i])

    # Processo
    navegador.find_element(By.ID, "numero").send_keys(processos_df["Processo"][i])

    # Selecionando o botão de pesquisa
    navegador.find_element(By.XPATH, '//*[@id="formulario"]/div/button').click()

    # Tratando o alerta

    from selenium.webdriver.common.alert import Alert
    import time
    alerta = Alert(navegador)
    alerta.accept()

    time.sleep(6)
    alerta = Alert(navegador)
    text = alerta.text
    alerta.accept()

    if "Nenhum processo encontrado!" in text:
        processos_df.loc[i,"Status"] = "Não Encontrado"
    else:
        processos_df.loc[i,"Status"] = "Encontrado"
    navegador.close()
    navegador.switch_to.window(primeira_aba)

In [185]:
processos_df.to_excel("Processos_Atualizado.xlsx")
display(processos_df)

,Nome,Advogado,Processo,Cidade,Status
0,Lira,Alon Lawyer,PC6592,Distrito Federal,Encontrado
1,João,Lawyer Alon,EB3792,Rio de Janeiro,Não Encontrado
2,Amanda,Amanda mesmo,MM1043,Rio de Janeiro,Encontrado
3,Carol,Amanda,PC5197,São Paulo,Encontrado
